In [1]:
import numpy as np
from sklearn import datasets
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.model_selection import KFold, GridSearchCV

In [2]:
# 1. Загружаем
newsgroups = datasets.fetch_20newsgroups(
    subset='all',
    categories=['alt.atheism', 'sci.space']
)
X_texts = newsgroups.data
y = newsgroups.target

# 2. Вычисляем TF-IDF по всему
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(X_texts)

# 3. Подбираем C
C_range = np.power(10.0, np.arange(-5, 6))
param_grid = {'C': C_range}

cv = KFold(n_splits=5, shuffle=True, random_state=241)
svc = SVC(kernel='linear', random_state=241)
grid_search = GridSearchCV(svc, param_grid, scoring='accuracy', cv=cv)
grid_search.fit(X, y)

best_C = grid_search.best_params_['C']

# 4. Обучаем SVM на всей выборке с лучшим C
clf = SVC(kernel='linear', C=best_C, random_state=241)
clf.fit(X, y)

# 5. Топ-10 слов по модулю весов
feature_names = vectorizer.get_feature_names_out()
coef = clf.coef_.toarray().flatten()
top_indices = np.argsort(np.abs(coef))[-10:]
top_words = [feature_names[i] for i in top_indices]
top_words_sorted = sorted(top_words, key=str.lower)

answer = ",".join(top_words_sorted)


print("Лучший C:", best_C)
print("Топ-10 слов:", answer)

Лучший C: 1.0
Топ-10 слов: atheism,atheists,bible,god,keith,moon,religion,sci,sky,space


In [3]:
with open("answer.txt", "w", encoding="utf-8") as f:
    f.write(answer)